# Pandas for AI Students: A Step-by-Step Tutorial

**Goal:** Learn to load, explore, clean, transform, and prepare tabular data with Pandas, then feed it into a machine-learning model.

**Prerequisites:** Basic Python and NumPy.

## Table of Contents
1. Why Pandas?
2. Installation and Import
3. Series
4. DataFrame Basics and Our Practice Dataset
5. Inspecting Data
6. Reading and Writing Files
7. Selecting Data: `[]`, `loc`, `iloc`, Boolean Filters, `query`
8. Creating and Modifying Columns
9. Handling Missing Values
10. Duplicates and String Cleaning
11. Sorting and Ranking
12. GroupBy and Aggregation
13. Pivot Tables and Crosstabs
14. Combining DataFrames: `concat`, `merge`, `join`
15. Reshaping: `melt`, `pivot`, `stack`
16. Time Series
17. Categorical Data, Binning, and Encoding
18. Plotting with Pandas
19. Performance Tips
20. AI Mini-Projects
21. Exercises
22. Cheat Sheet

## 1. Why Pandas?

**Pandas** is the standard Python library for working with **tabular data** (rows and columns, like Excel sheets or SQL tables). Real-world AI projects spend most of their time on data preparation, and Pandas is the main tool for that.

| NumPy | Pandas |
|---|---|
| Homogeneous n-dimensional arrays | Heterogeneous tables (each column has its own type) |
| Integer positions only | Labeled rows (index) and columns |
| No built-in missing-value handling | Rich missing-data support (`NaN`) |
| Great for math | Great for cleaning, grouping, joining, time series |

### Two core data structures
- **Series:** a 1D labeled array (a single column).
- **DataFrame:** a 2D labeled table (a collection of Series that share an index).

### A typical ML data workflow
`Load -> Explore -> Clean -> Feature engineering -> Encode/Scale -> Convert to NumPy -> Train model`

## 2. Installation and Import

```bash
pip install pandas numpy matplotlib
```

By convention we import Pandas as `pd`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

print("Pandas version:", pd.__version__)

# Display options (optional but helpful)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)
pd.set_option("display.precision", 3)

## 3. Series

A **Series** is a 1D array with an **index** (labels).

In [ ]:
s = pd.Series([10, 20, 30, 40], index=["a", "b", "c", "d"], name="values")
print(s)
print()
print("values:", s.values)       # underlying NumPy array
print("index :", s.index)
print("dtype :", s.dtype)

### Accessing elements
Use labels with `.loc` and integer positions with `.iloc`.

In [ ]:
print("s['b']        :", s["b"])
print("s.loc['b']    :", s.loc["b"])
print("s.iloc[1]     :", s.iloc[1])
print("s['a':'c']    :\n", s["a":"c"])        # label slices INCLUDE the end label
print("s.iloc[0:2]   :\n", s.iloc[0:2])        # integer slices EXCLUDE the end

### Vectorized operations and alignment
Operations are element-wise, and Pandas **aligns by index label** automatically.

In [ ]:
a = pd.Series([1, 2, 3], index=["x", "y", "z"])
b = pd.Series([10, 20, 30], index=["y", "z", "w"])

print("a * 2:\n", a * 2)
print("\na + b (aligned by label, unmatched -> NaN):\n", a + b)
print("\na.add(b, fill_value=0):\n", a.add(b, fill_value=0))

### Useful Series methods

In [ ]:
colors = pd.Series(["red", "blue", "red", "green", "blue", "red"])

print("unique     :", colors.unique())
print("nunique    :", colors.nunique())
print("value_counts:\n", colors.value_counts())
print("normalized :\n", colors.value_counts(normalize=True).round(2))

nums = pd.Series([3, 7, 1, 9, 4])
print("\nsum/mean/std:", nums.sum(), nums.mean(), round(nums.std(), 3))
print("max index   :", nums.idxmax())
print("describe:\n", nums.describe())

## 4. DataFrame Basics and Our Practice Dataset

### 4.1 Creating a DataFrame
From a dictionary of columns:

In [ ]:
df_small = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Diana"],
    "age": [25, 32, 47, 19],
    "city": ["Paris", "London", "Paris", "Berlin"],
    "salary": [52000.0, 61000.0, 83000.0, 28000.0],
})
df_small

From a list of dictionaries (one per row), a NumPy array, or a list of lists:

In [ ]:
rows = [{"x": 1, "y": 2}, {"x": 3, "y": 4}, {"x": 5}]      # missing key -> NaN
print(pd.DataFrame(rows))

arr = np.arange(6).reshape(3, 2)
print(pd.DataFrame(arr, columns=["col_a", "col_b"], index=["r1", "r2", "r3"]))

### 4.2 Our practice dataset
Real datasets need downloads, so we generate a **synthetic Titanic-style passenger dataset** with realistic problems: missing values, skewed fares, text columns, and a binary target `survived`. We will use it throughout the notebook.

In [ ]:
def make_passengers(n=400, seed=0):
    rng = np.random.default_rng(seed)

    sex = rng.choice(["male", "female"], size=n, p=[0.64, 0.36])
    pclass = rng.choice([1, 2, 3], size=n, p=[0.25, 0.20, 0.55])
    age = np.clip(rng.normal(30, 13, n), 1, 80).round(0)
    sibsp = rng.poisson(0.5, n)
    parch = rng.poisson(0.4, n)
    fare_mu = np.where(pclass == 1, 4.2, np.where(pclass == 2, 3.0, 2.2))
    fare = np.round(rng.lognormal(mean=fare_mu, sigma=0.5), 2)
    embarked = rng.choice(["S", "C", "Q"], size=n, p=[0.70, 0.20, 0.10])

    last_names = ["Smith", "Brown", "Taylor", "Wilson", "Davies", "Evans", "Thomas", "Johnson",
                  "Roberts", "Walker", "Wright", "Robinson", "Thompson", "White", "Hughes"]
    male_first = ["John", "William", "James", "George", "Henry", "Thomas", "Charles"]
    female_first = ["Mary", "Anna", "Emma", "Alice", "Helen", "Margaret", "Sarah"]

    titles, names = [], []
    for i in range(n):
        if sex[i] == "male":
            title = "Master" if age[i] < 13 else "Mr"
            first = rng.choice(male_first)
        else:
            title = "Miss" if age[i] < 18 else rng.choice(["Mrs", "Miss"])
            first = rng.choice(female_first)
        titles.append(title)
        names.append(f"{rng.choice(last_names)}, {title}. {first}")

    logit = (-0.6 + 2.2 * (sex == "female") - 0.8 * (pclass - 2)
             - 0.025 * (age - 30) - 0.15 * sibsp)
    survived = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)

    cabin = [f"C{rng.integers(1, 120)}" if (pclass[i] == 1 and rng.random() < 0.7) else None
             for i in range(n)]

    df = pd.DataFrame({
        "passenger_id": np.arange(1, n + 1),
        "name": names,
        "sex": sex,
        "age": age,
        "pclass": pclass,
        "sibsp": sibsp,
        "parch": parch,
        "fare": fare,
        "embarked": embarked,
        "cabin": cabin,
        "survived": survived,
    })

    # Inject missing values
    df.loc[rng.random(n) < 0.20, "age"] = np.nan
    df.loc[rng.random(n) < 0.01, "embarked"] = np.nan
    return df

df = make_passengers()
df.head()

## 5. Inspecting Data

Always inspect a new dataset before doing anything else.

In [ ]:
print("Shape  :", df.shape)            # (rows, columns)
print("Columns:", list(df.columns))
print("Index  :", df.index)
print()
df.head(3)

In [ ]:
df.tail(3)

In [ ]:
df.info()

In [ ]:
# Summary statistics for numeric columns
df.describe()

In [ ]:
# Include text columns as well
df.select_dtypes(exclude="number").describe()

In [ ]:
print("dtypes:\n", df.dtypes)
print("\nMissing values per column:\n", df.isna().sum())
print("\nMissing percentage:\n", (df.isna().mean() * 100).round(1))
print("\nTarget balance:\n", df["survived"].value_counts(normalize=True).round(3))
print("\nRandom sample:")
df.sample(3, random_state=1)

## 6. Reading and Writing Files

| Format | Read | Write |
|---|---|---|
| CSV | `pd.read_csv(path)` | `df.to_csv(path, index=False)` |
| Excel | `pd.read_excel(path)` (needs `openpyxl`) | `df.to_excel(path)` |
| JSON | `pd.read_json(path)` | `df.to_json(path)` |
| Parquet | `pd.read_parquet(path)` (needs `pyarrow`) | `df.to_parquet(path)` |
| SQL | `pd.read_sql(query, connection)` | `df.to_sql(name, connection)` |

You can also read from a URL: `pd.read_csv("https://example.com/data.csv")`.

In [ ]:
import os, tempfile

folder = tempfile.mkdtemp()
csv_path = os.path.join(folder, "passengers.csv")
json_path = os.path.join(folder, "passengers.json")

df.to_csv(csv_path, index=False)                  # index=False avoids an extra index column
df.head(5).to_json(json_path, orient="records", indent=2)

loaded = pd.read_csv(csv_path)
print("Reloaded shape:", loaded.shape)
print(pd.read_json(json_path).head(2))

### Useful `read_csv` options

In [ ]:
partial = pd.read_csv(
    csv_path,
    usecols=["passenger_id", "sex", "age", "fare", "survived"],   # load only some columns
    dtype={"sex": "category"},                                    # set dtypes while loading
    nrows=100,                                                    # read only the first 100 rows
    index_col="passenger_id",                                     # use a column as the index
)
print(partial.dtypes)
partial.head(3)

Other helpful arguments: `sep=";"`, `header=None`, `names=[...]`, `na_values=["?", "N/A"]`, `parse_dates=["date"]`, `skiprows=`, `encoding="utf-8"`, `chunksize=` (stream huge files).

## 7. Selecting Data

### 7.1 Selecting columns

In [ ]:
print(type(df["age"]))              # one column -> Series
print(type(df[["age", "fare"]]))    # list of columns -> DataFrame
df[["name", "age", "fare"]].head(3)

### 7.2 `loc` (label-based) and `iloc` (position-based)

- `df.loc[row_selector, column_selector]`: uses **labels**; slices are **inclusive**.
- `df.iloc[row_positions, column_positions]`: uses **integer positions**; slices are **exclusive** at the end.

In [ ]:
print("loc, one cell    :", df.loc[0, "name"])
print("iloc, one cell   :", df.iloc[0, 1])

print("\nloc rows 0-2, two columns (inclusive):")
print(df.loc[0:2, ["name", "age"]])

print("\niloc first 3 rows, first 3 columns (exclusive):")
print(df.iloc[0:3, 0:3])

print("\nLast row:")
print(df.iloc[-1])

### 7.3 Boolean filtering
Combine conditions with `&` (and), `|` (or), `~` (not). **Always wrap each condition in parentheses.**

In [ ]:
women = df[df["sex"] == "female"]
print("Women:", len(women))

first_class_women = df[(df["sex"] == "female") & (df["pclass"] == 1)]
print("First-class women:", len(first_class_women))

young_or_rich = df[(df["age"] < 10) | (df["fare"] > 200)]
print("Children or high fare:", len(young_or_rich))

not_southampton = df[~(df["embarked"] == "S")]
print("Not from S:", len(not_southampton))

### 7.4 `isin`, `between`, string and null filters

In [ ]:
print("isin       :", len(df[df["embarked"].isin(["C", "Q"])]))
print("between    :", len(df[df["age"].between(20, 30)]))
print("str.contains:", len(df[df["name"].str.contains("Mrs")]))
print("has cabin  :", len(df[df["cabin"].notna()]))
print("age missing:", len(df[df["age"].isna()]))

### 7.5 `query` (readable filtering)

In [ ]:
min_age = 18
adults = df.query("age >= @min_age and pclass == 1 and sex == 'female'")
print(adults[["name", "age", "fare", "survived"]].head())

### 7.6 Setting values safely
Use `.loc` to assign (never chain like `df[cond]["col"] = x`). When you derive a new DataFrame you plan to modify, call `.copy()`.

In [ ]:
tmp = df.copy()
tmp.loc[tmp["age"] > 70, "age"] = 70          # cap ages at 70
print("Max age now:", tmp["age"].max())

## 8. Creating and Modifying Columns

### 8.1 Vectorized arithmetic

In [ ]:
data = df.copy()

data["family_size"] = data["sibsp"] + data["parch"] + 1
data["is_alone"] = (data["family_size"] == 1).astype(int)
data["fare_per_person"] = data["fare"] / data["family_size"]
data["log_fare"] = np.log1p(data["fare"])        # NumPy functions work on columns

data[["fare", "family_size", "is_alone", "fare_per_person", "log_fare"]].head()

### 8.2 Conditional columns: `np.where`, `np.select`, `map`, `apply`

In [ ]:
# Binary condition
data["is_child"] = np.where(data["age"] < 13, 1, 0)

# Multiple conditions
conditions = [data["fare"] < 10, data["fare"] < 50]
choices = ["cheap", "medium"]
data["fare_level"] = np.select(conditions, choices, default="expensive")

# Map values with a dictionary
data["sex_code"] = data["sex"].map({"male": 0, "female": 1})

# apply: custom function on one column (slower than vectorized code!)
data["name_length"] = data["name"].apply(len)

# apply on rows with axis=1 (slowest, use only when necessary)
data["summary"] = data.apply(lambda r: f"{r['sex']}-class{r['pclass']}", axis=1)

data[["is_child", "fare_level", "sex_code", "name_length", "summary"]].head()

### 8.3 Rename, reorder, drop, change types

In [ ]:
tmp = data.rename(columns={"pclass": "ticket_class", "sibsp": "siblings_spouses"})
print(list(tmp.columns)[:6])

tmp = data.drop(columns=["name", "summary", "name_length"])     # drop columns
tmp = tmp.drop(index=[0, 1, 2])                                  # drop rows by index label
print("Shape after drops:", tmp.shape)

# Change dtype
tmp["pclass"] = tmp["pclass"].astype("int8")
tmp["sex"] = tmp["sex"].astype("category")
print(tmp[["pclass", "sex"]].dtypes)

# Insert a column at a specific position
tmp.insert(1, "constant", 1)
print(list(tmp.columns)[:4])

### 8.4 `assign` for chaining
`assign` returns a new DataFrame, which lets you build readable method chains.

In [ ]:
result = (
    df
    .assign(family_size=lambda d: d["sibsp"] + d["parch"] + 1,
            adult=lambda d: d["age"] >= 18)
    .query("family_size > 3")
    .loc[:, ["name", "family_size", "adult", "survived"]]
    .head()
)
result

## 9. Handling Missing Values

Missing values appear as `NaN` (or `None`/`NaT`). ML models usually cannot accept them, so you must **drop** or **impute** them.

### 9.1 Detect

In [ ]:
print(df.isna().sum())
print("\nRows with at least one missing value:", df.isna().any(axis=1).sum())
print("Rows with missing age:", df["age"].isna().sum())

### 9.2 Drop

In [ ]:
print("Original rows            :", len(df))
print("dropna() all columns     :", len(df.dropna()))
print("dropna(subset=['age'])   :", len(df.dropna(subset=["age"])))
print("dropna(thresh=...)       :", len(df.dropna(thresh=9)))     # keep rows with >= 9 non-null values
print("Columns after dropna(axis=1):", list(df.dropna(axis=1).columns))

### 9.3 Fill (impute)
Common strategies: constant, mean, median, mode, group-wise statistic, forward/backward fill (time series).

In [ ]:
d = df.copy()

# 1. Constant
d["cabin"] = d["cabin"].fillna("Unknown")

# 2. Mode (most frequent) for categorical columns
d["embarked"] = d["embarked"].fillna(d["embarked"].mode()[0])

# 3. Group-wise median for numeric columns: more accurate than a global median
d["age"] = d["age"].fillna(d.groupby(["pclass", "sex"])["age"].transform("median"))

print(d.isna().sum())

### 9.4 Interpolation and forward fill (ordered data)

In [ ]:
ts = pd.Series([1.0, np.nan, np.nan, 4.0, np.nan, 6.0])
print("original   :", ts.tolist())
print("ffill      :", ts.ffill().tolist())
print("bfill      :", ts.bfill().tolist())
print("interpolate:", ts.interpolate().tolist())

> **Tip for ML:** Fit imputation statistics (mean, median, ...) on the **training set only** and reuse them for validation and test data, otherwise you leak information.
>
> **Tip:** Sometimes missingness itself is informative. Add an indicator column such as `df["age_missing"] = df["age"].isna().astype(int)` before imputing.

## 10. Duplicates and String Cleaning

### 10.1 Duplicates

In [ ]:
dup_demo = pd.concat([df, df.sample(5, random_state=0)], ignore_index=True)   # add 5 duplicate rows

print("Total rows           :", len(dup_demo))
print("Duplicated rows      :", dup_demo.duplicated().sum())
print("Duplicates by id only:", dup_demo.duplicated(subset=["passenger_id"]).sum())

clean = dup_demo.drop_duplicates()                      # keeps the first occurrence by default
print("After drop_duplicates:", len(clean))

### 10.2 String methods (`.str` accessor)
Text columns need cleaning before use. The `.str` accessor applies string functions to a whole column.

In [ ]:
messy = pd.Series(["  Alice SMITH ", "bob jones", "CHARLIE  brown", None, "diana-Prince"])

print("lower  :", messy.str.lower().tolist())
print("strip  :", messy.str.strip().tolist())
print("title  :", messy.str.strip().str.title().tolist())
print("len    :", messy.str.len().tolist())
print("replace:", messy.str.replace("-", " ", regex=False).tolist())
print("split  :", messy.str.strip().str.split(" ").tolist())

### 10.3 Extracting features with regular expressions
Extract the **title** (Mr, Mrs, Miss, Master) from the name. This is a classic feature-engineering trick.

In [ ]:
names = df["name"]
print(names.head(3).tolist())

df_feat = df.copy()
df_feat["title"] = names.str.extract(r",\s*([A-Za-z]+)\.", expand=False)
df_feat["last_name"] = names.str.split(",").str[0]

print(df_feat[["name", "title", "last_name"]].head())
print("\n", df_feat["title"].value_counts())

## 11. Sorting and Ranking

In [ ]:
print("Top 5 fares:")
print(df.sort_values("fare", ascending=False)[["name", "pclass", "fare"]].head())

print("\nSort by two columns (class asc, fare desc):")
print(df.sort_values(["pclass", "fare"], ascending=[True, False])[["pclass", "fare"]].head())

print("\nnlargest / nsmallest (faster for top-k):")
print(df.nlargest(3, "age")[["name", "age"]])
print(df.nsmallest(3, "age")[["name", "age"]])

df_ranked = df.assign(fare_rank=df["fare"].rank(ascending=False, method="min"))
print("\nRanks:")
print(df_ranked[["fare", "fare_rank"]].head())

# Sort by index
print("\nsort_index:")
print(df.sample(4, random_state=0).sort_index().index.tolist())

## 12. GroupBy and Aggregation

`groupby` follows the **split -> apply -> combine** pattern: split data into groups, compute something per group, combine the results.

In [ ]:
# One aggregation
print(df.groupby("sex")["survived"].mean())

# Several aggregations
print(df.groupby("pclass")["fare"].agg(["count", "mean", "median", "max"]))

### Named aggregations (clean and explicit)

In [ ]:
summary = df.groupby(["pclass", "sex"]).agg(
    passengers=("passenger_id", "count"),
    survival_rate=("survived", "mean"),
    avg_age=("age", "mean"),
    median_fare=("fare", "median"),
).round(3)

summary

### `reset_index`, `transform`, `filter`
- `reset_index()` turns group keys back into regular columns.
- `transform` returns a result **with the same shape as the input** (great for group-wise features).
- `filter` keeps or removes entire groups.

In [ ]:
flat = summary.reset_index()
print(flat.head(3))

# Group-wise feature: how does each fare compare with the class average?
d = df.copy()
d["class_avg_fare"] = d.groupby("pclass")["fare"].transform("mean")
d["fare_ratio"] = d["fare"] / d["class_avg_fare"]
print(d[["pclass", "fare", "class_avg_fare", "fare_ratio"]].head())

# Keep only embarkation ports with more than 50 passengers
big_ports = df.groupby("embarked").filter(lambda g: len(g) > 50)
print("\nPorts kept:", big_ports["embarked"].unique())

### Iterating over groups

In [ ]:
for cls, group in df.groupby("pclass"):
    print(f"Class {cls}: {len(group)} passengers, survival = {group['survived'].mean():.2%}")

## 13. Pivot Tables and Crosstabs

### `pivot_table`
Like Excel pivot tables: choose rows, columns, values, and an aggregation.

In [ ]:
pivot = pd.pivot_table(
    df,
    values="survived",
    index="pclass",
    columns="sex",
    aggfunc="mean",
    margins=True,           # add totals row and column
).round(3)
pivot

### `crosstab` (frequency tables)

In [ ]:
print(pd.crosstab(df["pclass"], df["survived"]))
print()
print(pd.crosstab(df["pclass"], df["survived"], normalize="index").round(3))   # row percentages

## 14. Combining DataFrames

### 14.1 `concat` (stack along an axis)

In [ ]:
part1 = df.iloc[:3][["passenger_id", "name"]]
part2 = df.iloc[3:6][["passenger_id", "name"]]

print("Vertical (axis=0):")
print(pd.concat([part1, part2], ignore_index=True))

extra = pd.DataFrame({"score": [0.1, 0.2, 0.3]}, index=part1.index)
print("\nHorizontal (axis=1):")
print(pd.concat([part1, extra], axis=1))

### 14.2 `merge` (SQL-style joins)
`how` can be `inner`, `left`, `right`, or `outer`.

In [ ]:
class_info = pd.DataFrame({
    "pclass": [1, 2, 3],
    "class_name": ["First", "Second", "Third"],
    "deck": ["A-C", "D-E", "F-G"],
})

merged = df.merge(class_info, on="pclass", how="left")
print(merged[["name", "pclass", "class_name", "deck"]].head())

In [ ]:
left = pd.DataFrame({"id": [1, 2, 3, 4], "left_val": ["a", "b", "c", "d"]})
right = pd.DataFrame({"id": [3, 4, 5, 6], "right_val": ["x", "y", "z", "w"]})

for how in ["inner", "left", "right", "outer"]:
    print(f"--- {how} join ---")
    print(left.merge(right, on="id", how=how))

Different key names: `left.merge(right, left_on="a", right_on="b")`. Join on index: `left.join(right, how="left")`. Add `validate="m:1"` to `merge` to catch unexpected duplicate keys, and `indicator=True` to see which side each row came from.

## 15. Reshaping: `melt`, `pivot`, `stack`

- **Wide format:** one row per entity, one column per measurement.
- **Long (tidy) format:** one row per observation. Plotting libraries and many ML tools prefer long format.

In [ ]:
wide = pd.DataFrame({
    "student": ["Ann", "Ben", "Cara"],
    "math": [90, 70, 85],
    "physics": [80, 75, 95],
    "chemistry": [85, 60, 90],
})
print("Wide:")
print(wide)

long = wide.melt(id_vars="student", var_name="subject", value_name="score")
print("\nLong (melt):")
print(long)

back = long.pivot(index="student", columns="subject", values="score")
print("\nBack to wide (pivot):")
print(back)

In [ ]:
# stack / unstack move column labels into the index and back
stacked = wide.set_index("student").stack()
print(stacked.head(4))
print(stacked.unstack().head(2))

## 16. Time Series

Pandas has excellent support for dates and times.

### 16.1 Dates and the `.dt` accessor

In [ ]:
dates = pd.to_datetime(["2024-01-15", "2024-02-20", "2024-03-25"])
print(dates)

s = pd.Series(pd.to_datetime(["2024-03-15 14:30", "2024-07-04 09:00", "2024-12-25 18:45"]))
print("year     :", s.dt.year.tolist())
print("month    :", s.dt.month.tolist())
print("dayofweek:", s.dt.dayofweek.tolist(), "(Monday=0)")
print("day name :", s.dt.day_name().tolist())
print("hour     :", s.dt.hour.tolist())
print("Difference:", (s.iloc[1] - s.iloc[0]).days, "days")

### 16.2 A time-indexed dataset
We simulate 120 days of store sales with a weekly pattern and an upward trend.

In [ ]:
rng = np.random.default_rng(42)
idx = pd.date_range("2024-01-01", periods=120, freq="D")

weekly_pattern = np.where(idx.dayofweek >= 5, 1.4, 1.0)         # weekends are busier
sales = (200 + np.arange(120) * 0.8) * weekly_pattern + rng.normal(0, 15, 120)

ts = pd.DataFrame({"sales": sales.round(1)}, index=idx)
ts.index.name = "date"
ts.head()

### 16.3 Slicing by date, resampling, rolling windows

In [ ]:
print("January mean sales:", ts.loc["2024-01", "sales"].mean().round(1))
print("Feb 10 - Feb 14:")
print(ts.loc["2024-02-10":"2024-02-14"])

# Resample: change the frequency (like groupby for time)
weekly = ts["sales"].resample("W").sum()
monthly = ts["sales"].resample("MS").mean().round(1)
print("\nWeekly sums (first 3):\n", weekly.head(3))
print("\nMonthly means:\n", monthly)

In [ ]:
# Rolling (moving) statistics
ts["ma_7"] = ts["sales"].rolling(window=7).mean()
ts["ma_30"] = ts["sales"].rolling(window=30).mean()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ts.index, ts["sales"], alpha=0.4, label="Daily sales")
ax.plot(ts.index, ts["ma_7"], label="7-day average")
ax.plot(ts.index, ts["ma_30"], label="30-day average", lw=2)
ax.set_title("Sales with Moving Averages")
ax.legend()
plt.show()

### 16.4 Lag features, differences, and growth (key for forecasting)

In [ ]:
feat = ts[["sales"]].copy()
feat["lag_1"] = feat["sales"].shift(1)              # yesterday's sales
feat["lag_7"] = feat["sales"].shift(7)              # same weekday last week
feat["diff_1"] = feat["sales"].diff()               # change from yesterday
feat["pct_change"] = feat["sales"].pct_change()     # percentage change
feat["day_of_week"] = feat.index.dayofweek
feat["is_weekend"] = (feat.index.dayofweek >= 5).astype(int)
feat["month"] = feat.index.month

feat.iloc[7:12].round(3)

> **Important:** For time series, never shuffle randomly when splitting. Train on the **past** and test on the **future**, for example `train = feat.iloc[:100]` and `test = feat.iloc[100:]`.

## 17. Categorical Data, Binning, and Encoding

ML models need numbers. These are the main ways to turn categories into numeric features.

### 17.1 The `category` dtype
Saves memory and speeds up grouping for columns with few unique values.

In [ ]:
cat = df["embarked"].astype("category")
print(cat.cat.categories.tolist())
print("Integer codes:", cat.cat.codes.head().tolist())

mem_object = df["embarked"].memory_usage(deep=True)
mem_cat = cat.memory_usage(deep=True)
print(f"Memory: object = {mem_object} bytes, category = {mem_cat} bytes")

### 17.2 One-hot encoding with `get_dummies`
Use for **nominal** categories (no natural order). `drop_first=True` avoids redundant columns (the dummy-variable trap).

In [ ]:
encoded = pd.get_dummies(df[["sex", "embarked", "pclass"]], columns=["sex", "embarked"],
                         drop_first=True, dtype=int)
encoded.head()

### 17.3 Ordinal encoding with a mapping
Use for **ordered** categories (small < medium < large).

In [ ]:
sizes = pd.Series(["small", "large", "medium", "small", "large"])
order = {"small": 0, "medium": 1, "large": 2}
print(sizes.map(order).tolist())

### 17.4 Binning continuous values: `cut` and `qcut`
- `pd.cut`: bins with **chosen edges**.
- `pd.qcut`: bins with **equal numbers of samples** (quantiles).

In [ ]:
age_group = pd.cut(df["age"], bins=[0, 12, 18, 40, 60, 100],
                   labels=["child", "teen", "adult", "middle", "senior"])
print(age_group.value_counts().sort_index())

fare_quartile = pd.qcut(df["fare"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
print("\n", fare_quartile.value_counts().sort_index())

print("\nSurvival by age group:")
print(df.groupby(age_group, observed=True)["survived"].mean().round(3))

### 17.5 Rare-category grouping
Merge infrequent categories to reduce noise.

In [ ]:
titles = df["name"].str.extract(r",\s*([A-Za-z]+)\.", expand=False)
counts = titles.value_counts()
rare = counts[counts < 50].index
titles_grouped = titles.where(~titles.isin(rare), "Other")
print(titles_grouped.value_counts())

## 18. Plotting with Pandas

Pandas wraps Matplotlib: call `.plot()` on a Series or DataFrame.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

# Histogram
df["age"].plot(kind="hist", bins=25, ax=axes[0, 0], edgecolor="k", title="Age distribution")

# Bar chart from a groupby
df.groupby("pclass")["survived"].mean().plot(kind="bar", ax=axes[0, 1], color="teal",
                                              title="Survival rate by class", rot=0)
axes[0, 1].set_ylabel("Survival rate")

# Box plot
df.boxplot(column="fare", by="pclass", ax=axes[1, 0])
axes[1, 0].set_title("Fare by class")
axes[1, 0].set_yscale("log")
fig.suptitle("")

# Scatter plot colored by target
df.plot(kind="scatter", x="age", y="fare", c="survived", cmap="coolwarm", alpha=0.6,
        ax=axes[1, 1], title="Age vs fare (color = survived)")
axes[1, 1].set_yscale("log")

fig.tight_layout()
plt.show()

In [ ]:
# Stacked bar chart from a crosstab
ct = pd.crosstab(df["pclass"], df["survived"])
ct.plot(kind="bar", stacked=True, figsize=(6, 4), rot=0, color=["#e63946", "#2a9d8f"])
plt.title("Survived (1) vs not survived (0) by class")
plt.ylabel("Passengers")
plt.show()

## 19. Performance Tips

1. **Vectorize**: avoid `apply` or Python loops when a column-wise operation exists.
2. Use the **`category`** dtype for low-cardinality text columns.
3. Use **smaller numeric types** (`int8`, `float32`) for large data.
4. Read only the **columns you need** (`usecols`) and process large files in **chunks**.
5. Prefer `df.loc[mask, col] = value` over loops.

In [ ]:
import time

big = pd.DataFrame({"x": np.random.default_rng(0).random(300_000)})

t0 = time.perf_counter()
slow = big["x"].apply(lambda v: v * 2 + 1)
t_apply = time.perf_counter() - t0

t0 = time.perf_counter()
fast = big["x"] * 2 + 1
t_vec = time.perf_counter() - t0

print(f"apply     : {t_apply * 1000:.1f} ms")
print(f"vectorized: {t_vec * 1000:.2f} ms")
print(f"Speed-up  : ~{t_apply / t_vec:.0f}x")
print("Same result:", np.allclose(slow, fast))

In [ ]:
# Memory optimization
mem_before = df.memory_usage(deep=True).sum()

opt = df.copy()
opt["sex"] = opt["sex"].astype("category")
opt["embarked"] = opt["embarked"].astype("category")
opt["pclass"] = opt["pclass"].astype("int8")
opt["sibsp"] = opt["sibsp"].astype("int8")
opt["parch"] = opt["parch"].astype("int8")
opt["survived"] = opt["survived"].astype("int8")
opt["fare"] = opt["fare"].astype("float32")

mem_after = opt.memory_usage(deep=True).sum()
print(f"Memory before: {mem_before / 1024:.1f} KB")
print(f"Memory after : {mem_after / 1024:.1f} KB  ({1 - mem_after / mem_before:.0%} smaller)")

## 20. AI Mini-Projects

### Project 1: Exploratory Data Analysis (EDA)
Before modeling, ask questions of the data.

In [ ]:
df = make_passengers()

print("Overall survival rate:", round(df["survived"].mean(), 3))

print("\nSurvival by sex:")
print(df.groupby("sex")["survived"].agg(["mean", "count"]).round(3))

print("\nSurvival by class and sex:")
print(pd.pivot_table(df, values="survived", index="pclass", columns="sex", aggfunc="mean").round(3))

print("\nCorrelation of numeric features with survival:")
num_cols = ["age", "pclass", "sibsp", "parch", "fare", "survived"]
print(df[num_cols].corr()["survived"].drop("survived").round(3).sort_values())

In [ ]:
# Correlation heatmap
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols))); ax.set_xticklabels(num_cols, rotation=45, ha="right")
ax.set_yticks(range(len(num_cols))); ax.set_yticklabels(num_cols)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax)
ax.set_title("Correlation Matrix")
plt.show()

### Project 2: Outlier detection with the IQR rule
A value is an outlier if it lies below `Q1 - 1.5*IQR` or above `Q3 + 1.5*IQR`.

In [ ]:
q1, q3 = df["fare"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

outliers = df[(df["fare"] < lower) | (df["fare"] > upper)]
print(f"Q1 = {q1:.2f}, Q3 = {q3:.2f}, IQR = {iqr:.2f}")
print(f"Valid range: [{lower:.2f}, {upper:.2f}]")
print(f"Outliers: {len(outliers)} rows ({len(outliers) / len(df):.1%})")

# Options: remove, clip (winsorize), or transform (log)
df["fare_clipped"] = df["fare"].clip(lower=max(lower, 0), upper=upper)
df["fare_log"] = np.log1p(df["fare"])
print("\nSkewness before/after log:", round(df["fare"].skew(), 2), "->", round(df["fare_log"].skew(), 2))

### Project 3: A complete preprocessing pipeline -> NumPy -> model

We combine everything: clean, engineer features, encode, split, scale, and train a **logistic regression written in NumPy**.

In [ ]:
raw = make_passengers(n=1000, seed=7)

# ---------- 1. Clean ----------
data = raw.copy()
data["age_missing"] = data["age"].isna().astype(int)
data["age"] = data["age"].fillna(data.groupby(["pclass", "sex"])["age"].transform("median"))
data["embarked"] = data["embarked"].fillna(data["embarked"].mode()[0])
data["has_cabin"] = data["cabin"].notna().astype(int)

# ---------- 2. Feature engineering ----------
data["family_size"] = data["sibsp"] + data["parch"] + 1
data["is_alone"] = (data["family_size"] == 1).astype(int)
data["log_fare"] = np.log1p(data["fare"])
data["title"] = data["name"].str.extract(r",\s*([A-Za-z]+)\.", expand=False)

# ---------- 3. Encode ----------
data["sex"] = data["sex"].map({"male": 0, "female": 1})
data = pd.get_dummies(data, columns=["embarked", "title"], drop_first=True, dtype=int)

# ---------- 4. Select features ----------
drop_cols = ["passenger_id", "name", "cabin", "fare", "survived"]
X_df = data.drop(columns=drop_cols)
y = data["survived"]

print("Feature matrix shape:", X_df.shape)
print("Any missing values left?", X_df.isna().any().any())
print("Features:", list(X_df.columns))
X_df.head()

In [ ]:
# ---------- 5. Train/test split (random, reproducible) ----------
rng = np.random.default_rng(0)
perm = rng.permutation(len(X_df))
split = int(0.8 * len(X_df))
train_idx, test_idx = perm[:split], perm[split:]

X_train_df, X_test_df = X_df.iloc[train_idx], X_df.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# ---------- 6. Scale using TRAIN statistics only ----------
mean = X_train_df.mean()
std = X_train_df.std().replace(0, 1)          # avoid division by zero
X_train = ((X_train_df - mean) / std).to_numpy(dtype=float)
X_test = ((X_test_df - mean) / std).to_numpy(dtype=float)
y_train_np, y_test_np = y_train.to_numpy(), y_test.to_numpy()

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Train mean ~ 0:", np.abs(X_train.mean(axis=0)).max().round(6))

In [ ]:
# ---------- 7. Logistic regression with NumPy ----------
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

Xb_train = np.column_stack([np.ones(len(X_train)), X_train])
Xb_test = np.column_stack([np.ones(len(X_test)), X_test])

w = np.zeros(Xb_train.shape[1])
lr, epochs = 0.1, 1500
losses = []

for epoch in range(epochs):
    p = sigmoid(Xb_train @ w)
    grad = Xb_train.T @ (p - y_train_np) / len(y_train_np)
    w -= lr * grad
    eps = 1e-12
    losses.append(-np.mean(y_train_np * np.log(p + eps) + (1 - y_train_np) * np.log(1 - p + eps)))

train_acc = ((sigmoid(Xb_train @ w) > 0.5) == y_train_np).mean()
test_acc = ((sigmoid(Xb_test @ w) > 0.5) == y_test_np).mean()
baseline = max(y_test_np.mean(), 1 - y_test_np.mean())

print(f"Train accuracy       : {train_acc:.3f}")
print(f"Test accuracy        : {test_acc:.3f}")
print(f"Majority baseline    : {baseline:.3f}")

# ---------- 8. Interpret: which features matter? ----------
importance = pd.Series(w[1:], index=X_df.columns).sort_values()
print("\nLargest negative weights:\n", importance.head(3).round(3))
print("\nLargest positive weights:\n", importance.tail(3).round(3))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(losses)
ax1.set_title("Training Loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Log loss")
importance.plot(kind="barh", ax=ax2, color="slateblue")
ax2.set_title("Learned Weights (standardized features)")
fig.tight_layout()
plt.show()

### Project 4: Feature engineering for time-series forecasting
Build lag and rolling features from the sales data and fit a simple linear model, respecting time order.

In [ ]:
rng = np.random.default_rng(42)
idx = pd.date_range("2024-01-01", periods=200, freq="D")
sales = (200 + np.arange(200) * 0.8) * np.where(idx.dayofweek >= 5, 1.4, 1.0) + rng.normal(0, 15, 200)
ts = pd.DataFrame({"sales": sales}, index=idx)

# Features use ONLY past information (shift avoids leakage)
ts["lag_1"] = ts["sales"].shift(1)
ts["lag_7"] = ts["sales"].shift(7)
ts["roll_mean_7"] = ts["sales"].shift(1).rolling(7).mean()
ts["is_weekend"] = (ts.index.dayofweek >= 5).astype(int)
ts = ts.dropna()

features = ["lag_1", "lag_7", "roll_mean_7", "is_weekend"]

# Time-ordered split: train on the past, test on the future
cut = int(len(ts) * 0.8)
train, test = ts.iloc[:cut], ts.iloc[cut:]

Xtr = np.column_stack([np.ones(len(train)), train[features].to_numpy()])
Xte = np.column_stack([np.ones(len(test)), test[features].to_numpy()])
coef, *_ = np.linalg.lstsq(Xtr, train["sales"].to_numpy(), rcond=None)

pred = Xte @ coef
mae = np.mean(np.abs(pred - test["sales"].to_numpy()))
naive_mae = np.mean(np.abs(test["lag_1"].to_numpy() - test["sales"].to_numpy()))
print(f"Linear model MAE      : {mae:.2f}")
print(f"Naive (yesterday) MAE : {naive_mae:.2f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(test.index, test["sales"], label="Actual")
ax.plot(test.index, pred, label="Predicted", linestyle="--")
ax.set_title("Forecast on the Held-out Future Period")
ax.legend()
plt.show()

### Project 5: A reusable data-quality report
A small helper that summarizes every column. Useful at the start of every project.

In [ ]:
def quality_report(frame):
    report = pd.DataFrame({
        "dtype": frame.dtypes.astype(str),
        "missing": frame.isna().sum(),
        "missing_%": (frame.isna().mean() * 100).round(1),
        "unique": frame.nunique(),
    })
    numeric = frame.select_dtypes("number")
    report["min"] = numeric.min()
    report["max"] = numeric.max()
    return report

quality_report(make_passengers())

## 21. Exercises

Use `df = make_passengers()` (400 rows) unless a different dataset is specified. Try each exercise yourself before looking at the solutions.

1. How many passengers are in each class? What percentage does each class represent?
2. Find the 5 oldest passengers and show only `name`, `age`, and `pclass`.
3. Compute the average fare for each combination of `pclass` and `embarked`.
4. Fill missing `age` values with the **overall median**, then create a column `age_group` using bins `[0, 18, 35, 60, 100]` with labels `["youth", "young_adult", "adult", "senior"]`.
5. What fraction of women survived versus men? Express the answer as a one-row DataFrame.
6. Create a DataFrame `students` with columns `name`, `math`, `physics`. Add a `total` column and a `grade` column that is `"A"` if total >= 170, `"B"` if >= 140, otherwise `"C"`.
7. Merge two DataFrames: `orders = {order_id: [1,2,3,4], customer_id: [10,20,10,30]}` and `customers = {customer_id: [10,20,40], name: ["Ann","Bob","Dan"]}`. Show the result of a left join and count orders with no matching customer.
8. Convert the wide table `{"city": ["A","B"], "2022": [100,200], "2023": [110,190]}` to long format with columns `city`, `year`, `population`.
9. Using `ts` from Project 4, compute the **average sales per day of week** and identify the best day.
10. Write a function `iqr_outliers(series)` that returns a boolean mask of outliers, and use it to count outliers in `fare` for each class.

In [ ]:
# Write your solutions here

## Solutions

In [ ]:
df = make_passengers()

# 1
counts = df["pclass"].value_counts().sort_index()
print(pd.DataFrame({"count": counts, "percent": (counts / len(df) * 100).round(1)}))

# 2
print(df.nlargest(5, "age")[["name", "age", "pclass"]])

# 3
print(df.pivot_table(values="fare", index="pclass", columns="embarked", aggfunc="mean").round(2))

In [ ]:
# 4
d = df.copy()
d["age"] = d["age"].fillna(d["age"].median())
d["age_group"] = pd.cut(d["age"], bins=[0, 18, 35, 60, 100],
                        labels=["youth", "young_adult", "adult", "senior"])
print(d["age_group"].value_counts())

# 5
rates = df.groupby("sex")["survived"].mean().round(3)
print(rates.to_frame().T)

# 6
students = pd.DataFrame({"name": ["Ann", "Ben", "Cara"], "math": [90, 60, 75], "physics": [85, 70, 50]})
students["total"] = students["math"] + students["physics"]
students["grade"] = np.select([students["total"] >= 170, students["total"] >= 140], ["A", "B"], default="C")
print(students)

In [ ]:
# 7
orders = pd.DataFrame({"order_id": [1, 2, 3, 4], "customer_id": [10, 20, 10, 30]})
customers = pd.DataFrame({"customer_id": [10, 20, 40], "name": ["Ann", "Bob", "Dan"]})
joined = orders.merge(customers, on="customer_id", how="left")
print(joined)
print("Orders without a customer:", joined["name"].isna().sum())

# 8
wide = pd.DataFrame({"city": ["A", "B"], "2022": [100, 200], "2023": [110, 190]})
print(wide.melt(id_vars="city", var_name="year", value_name="population"))

# 9
rng = np.random.default_rng(42)
idx = pd.date_range("2024-01-01", periods=120, freq="D")
ts9 = pd.DataFrame({"sales": (200 + np.arange(120) * 0.8) * np.where(idx.dayofweek >= 5, 1.4, 1.0)
                    + rng.normal(0, 15, 120)}, index=idx)
by_day = ts9.groupby(ts9.index.day_name())["sales"].mean().round(1)
print(by_day.sort_values(ascending=False))
print("Best day:", by_day.idxmax())

# 10
def iqr_outliers(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)

print(df.groupby("pclass")["fare"].apply(lambda s: iqr_outliers(s).sum()))

## 22. Cheat Sheet

| Task | Code |
|---|---|
| Create DataFrame | `pd.DataFrame({"a": [1, 2], "b": [3, 4]})` |
| Read / write CSV | `pd.read_csv(path)`, `df.to_csv(path, index=False)` |
| Quick look | `df.head()`, `df.tail()`, `df.sample(5)` |
| Structure | `df.shape`, `df.columns`, `df.dtypes`, `df.info()` |
| Statistics | `df.describe()`, `df["c"].value_counts()`, `df.corr(numeric_only=True)` |
| Select columns | `df["a"]`, `df[["a", "b"]]` |
| Select rows | `df.loc[label_rows, cols]`, `df.iloc[pos_rows, pos_cols]` |
| Filter | `df[(df.a > 1) & (df.b == "x")]`, `df.query("a > 1")` |
| New column | `df["c"] = df.a + df.b`, `np.where(cond, x, y)` |
| Rename / drop | `df.rename(columns={...})`, `df.drop(columns=[...])` |
| Change type | `df["a"].astype("float32")`, `pd.to_datetime(df["d"])` |
| Missing values | `df.isna().sum()`, `df.dropna()`, `df.fillna(value)` |
| Duplicates | `df.duplicated()`, `df.drop_duplicates()` |
| Strings | `df["s"].str.lower()`, `.str.contains()`, `.str.extract()` |
| Sort | `df.sort_values("a", ascending=False)`, `df.nlargest(5, "a")` |
| GroupBy | `df.groupby("g")["v"].agg(["mean", "count"])` |
| Group feature | `df.groupby("g")["v"].transform("mean")` |
| Pivot / crosstab | `df.pivot_table(...)`, `pd.crosstab(a, b)` |
| Combine | `pd.concat([a, b])`, `a.merge(b, on="key", how="left")` |
| Reshape | `df.melt(...)`, `df.pivot(...)` |
| Time series | `df.resample("W").sum()`, `s.rolling(7).mean()`, `s.shift(1)` |
| Encode | `pd.get_dummies(df, drop_first=True)`, `pd.cut`, `pd.qcut` |
| To NumPy | `df.to_numpy()`, `df["a"].to_numpy()` |
| Plot | `df["a"].plot(kind="hist")`, `df.plot(x="a", y="b")` |

### Best practices
- **Inspect first** (`info`, `describe`, `isna().sum()`) and **never trust the data blindly**.
- Use **`.loc[mask, col] = value`** for assignment, and `.copy()` when you derive a DataFrame you intend to modify.
- **Vectorize**: avoid `apply(axis=1)` and loops on large data.
- Prefer **method chaining** with `assign`, `query`, and `pipe` for readable pipelines.
- Fit **imputers, scalers, and encoders on the training set only**, then apply them to validation and test sets (avoid data leakage).
- For time series, **split by time** and build features from the **past only**.
- Set `index=False` when saving CSV files unless the index is meaningful.
- Use the `category` dtype and compact numeric types to save memory.

### Next steps
- **scikit-learn** `Pipeline` and `ColumnTransformer` for production-ready preprocessing
- **Seaborn** for statistical visualization on DataFrames
- **Polars** or **Dask** for larger-than-memory or faster data processing
- **SQL** skills (`pd.read_sql`) for working with real databases

*Happy data wrangling!*